In [1]:
import sys
from pathlib import Path
sys.path[:0] = [str(Path.cwd().parent)]

import numpy as np
import matplotlib.pyplot as plt
from self_consistency_k import bdg_sc_full_k

In [2]:
corr_strings = [
    "F_onsite",
    "F_swave",
    "F_dwave",
    "F_px",
    "F_py",
    "Fuu_px",
    "Fuu_py",
    "Fdd_px",
    "Fdd_py",
]

In [3]:
def stable_config(out, atol=1e-6, rtol=0.1):
    stable = {}
    F_max = out.corr[np.argmax(np.abs(out.corr))]
    if np.abs(F_max) < atol:
        return stable
    for i, c in enumerate(out[1:-3]):
        if np.abs(c) > atol and np.abs(c) > rtol * np.abs(F_max):
            stable[corr_strings[i]] = c
    return stable

In [4]:
def same_stable_dict(d1, d2, atol=1e-6, rtol=1e-3):
    if d1.keys() != d2.keys():
        return False

    for k in d1:
        if not np.isclose(d1[k], d2[k], atol=atol, rtol=rtol):
            return False

    return True

In [5]:
initial_seeds = np.array([
    [0,0,0,0],  # normal state
    [0.1, -0.1, 0, 0],  # px 
    [0, 0, 0.1, -0.1],  # py
    [0.1, -0.1, 0.1, -0.1],  # px + py
    [0.1, -0.1, 0.1j, -0.1j],  # px + i*py
    [0.1, 0.1, -0.1, -0.1],  # d-wave
    [0.1+0.1, 0.1-0.1, -0.1, -0.1],  # d-wave + px
    [0.1, 0.1, -0.1+0.1, -0.1-0.1],  # d-wave + py
    [0.1, 0.1, 0.1, 0.1],  # s-wave
    [0.1+0.1, 0.1-0.1, 0.1, 0.1],  # s-wave + px
    [0.1, 0.1, 0.1+0.1, 0.1-0.1] # s-wave + py
], dtype=np.complex128)
seed_strings = [
    "normal state",
    "px", "py", "px+py", "px+i*py",
    "d-wave", "d-wave+px", "d-wave+py",
    "s-wave", "s-wave+px", "s-wave+py"
]

In [6]:
mu_arr = np.linspace(1.2, 4.0, 3)
T_arr_1 = np.linspace(0.01, 0.3, 2)
T_arr_2 = np.linspace(0.001, 0.05, 5)

free_tol = 0.01

In [7]:
t=1
Nx, Ny = 30, 30

V=1.5

atol = 1e-6
rtol = 1e-3
maxiter=1000

In [8]:
import pandas as pd

print_output = True
records = []

mu = mu_arr[1]

T_arr = T_arr_1 if mu < 1 else T_arr_2

for T in T_arr:
    best_free = np.inf
    configs = []
    print("====================================")
    print(f"mu={mu:.2f}, T={T:.4f}")
    print("====================================")
    for seed, seed_str in zip(initial_seeds, seed_strings):
        print(f"  Seed: {seed_str}")

        out = bdg_sc_full_k(
            t, mu, temperature=T,
            V=V, Nx=Nx, Ny=Ny,
            atol=atol, rtol=rtol,
            maxiter=maxiter,
            F_init=seed
        )
        if print_output == True:
            print(f"F_swave: {out.F_swave}, F_dwave: {out.F_dwave}")
            print(f"F_px: {out.F_px}, F_py: {out.F_py}")
            print(f"Free_energy = {out.free_energy}")
        stable = stable_config(out, atol=atol)

        print("------------------------------------------")
        if (out.free_energy < best_free 
            and np.abs(out.free_energy - best_free) > free_tol):

            best_free = out.free_energy
            configs = [{
                "stable": stable,
                "free": out.free_energy,
            }]

        elif np.abs(out.free_energy - best_free) <= free_tol:
            if len(stable) != 0 and not any(
                same_stable_dict(c["stable"], stable, atol=atol, rtol=rtol)
                for c in configs
            ):
                configs.append({
                    "stable": stable,
                    "free": out.free_energy,
                })

        print(configs)
        print("------------------------------------------")

    # Append one row per (mu, T)
    records.append({
        "mu": mu,
        "T": T,
        "best_free": best_free,
        "configs": configs
    })

# Create DataFrame
df = pd.DataFrame(records)

mu=2.60, T=0.0010
  Seed: normal state
F_swave: 0j, F_dwave: 0j
F_px: 0j, F_py: 0j
Free_energy = -2489.0240485102704
------------------------------------------
[{'stable': {}, 'free': np.float64(-2489.0240485102704)}]
------------------------------------------
  Seed: px
F_swave: -0.022361217949617297j, F_dwave: -8.134951046123717e-13j
F_px: (9.98694229084165e-09+0j), F_py: (1.3621983857743658e-17+0j)
Free_energy = -2492.3038360724654
------------------------------------------
[{'stable': {'F_onsite': np.complex128(0.029237243795326738j), 'F_swave': np.complex128(-0.022361217949617297j)}, 'free': np.float64(-2492.3038360724654)}]
------------------------------------------
  Seed: py
F_swave: -0.02236139608570436j, F_dwave: 5.649977014021701e-13j
F_px: (1.7816491012546953e-17+0j), F_py: (7.863512075622413e-09+0j)
Free_energy = -2492.303903397308
------------------------------------------
[{'stable': {'F_onsite': np.complex128(0.029237243795326738j), 'F_swave': np.complex128(-0.022361217

In [15]:
print(df[0:100])

    mu        T    best_free  \
0  2.6  0.00100 -2492.303836   
1  2.6  0.01325 -2492.294412   
2  2.6  0.02550 -2491.996680   
3  2.6  0.03775 -2491.092490   
4  2.6  0.05000 -2490.162784   

                                             configs  
0  [{'stable': {'F_onsite': 0.029237243795326738j...  
1  [{'stable': {'F_onsite': 0.029180500796333723j...  
2  [{'stable': {'F_onsite': 0.027312178768228904j...  
3  [{'stable': {'F_onsite': 0.0201940976592241j, ...  
4      [{'stable': {}, 'free': -2490.1627836426105}]  


In [16]:
def flatten_df(df):
    df_flat = df.explode("configs").reset_index(drop=True)

    # Split out the free energy and stable dict
    df_flat["free"] = df_flat["configs"].apply(
        lambda x: x.get("free") if isinstance(x, dict) else 0
    )
    df_flat["stable"] = df_flat["configs"].apply(
        lambda x: x.get("stable") if isinstance(x, dict) else {}
    )

    # Expand the stable dictionaries into columns
    stable_df = pd.DataFrame(df_flat["stable"].tolist())

    # Ensure all desired columns exist
    stable_df = stable_df.reindex(columns=corr_strings)

    # Combine everything
    df_flat = pd.concat(
        [
            df_flat.drop(columns=["configs", "stable"]),
            stable_df
        ],
        axis=1
    )
    df_flat.fillna(0, inplace=True)

    return df_flat

In [18]:
df_flat = flatten_df(df)
print(df_flat[0:100])

     mu        T    best_free         free            F_onsite  \
0   2.6  0.00100 -2492.303836 -2492.303836  0.000000+0.029237j   
1   2.6  0.00100 -2492.303836 -2492.303914  0.029237+0.000000j   
2   2.6  0.00100 -2492.303836 -2492.304267 -0.029239+0.000000j   
3   2.6  0.01325 -2492.294412 -2492.294412  0.000000+0.029181j   
4   2.6  0.01325 -2492.294412 -2492.293857 -0.022359+0.018747j   
5   2.6  0.01325 -2492.294412 -2492.294414  0.029181+0.000000j   
6   2.6  0.01325 -2492.294412 -2492.294424 -0.029181+0.000000j   
7   2.6  0.02550 -2491.996680 -2491.996680  0.000000+0.027312j   
8   2.6  0.02550 -2491.996680 -2491.996680 -0.020929+0.017548j   
9   2.6  0.02550 -2491.996680 -2491.996680  0.027312+0.000000j   
10  2.6  0.02550 -2491.996680 -2491.996680 -0.027312+0.000000j   
11  2.6  0.03775 -2491.092490 -2491.092490  0.000000+0.020194j   
12  2.6  0.03775 -2491.092490 -2491.092490 -0.015474+0.012975j   
13  2.6  0.03775 -2491.092490 -2491.092490  0.020194+0.000000j   
14  2.6  0

In [ ]:
# df_flat.to_parquet("results_flat.parquet", index=False)
df_flat.to_json("results.json")